# 01 - Dataset Preprocessing

Download Lichess standard rated games across multiple time periods (pre-AI and post-AI),
sample 200k games with fair ELO and period distribution, then extract:

1. **games.csv** - ELO, opening, winner, first moves, castling, game length, sacrifice count, eval volatility
2. **moves.csv** - eval, clock, piece, square, material, captures, checks, sacrifices (eval moves only)
3. **blunders.csv** - moves causing eval drops >= 200cp
4. **piece_squares.csv** - aggregated piece-square counts by period (for heatmaps)
5. **material_curve.csv** - average material at each ply by period

Designed to tell the story of how AI (AlphaZero 2017, Stockfish NNUE 2020) changed human chess.

## Configuration
Set the month to download and processing parameters below.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Time periods to sample (before and after AI breakthroughs)
PERIODS = [
    {"years": [2015, 2016], "label": "pre-ai"},
    {"years": [2018, 2019], "label": "early-post-ai"},
    {"years": [2021, 2022], "label": "nnue-era"},
    {"years": [2024],        "label": "modern"},
]

# Months per year to scan (None = all 12)
MONTH = None  # e.g. [1, 6, 12] for specific months

# Total games to collect (evenly split across period x ELO cells)
TOTAL_GAMES = 200_000

# ELO brackets for fair distribution
ELO_BRACKETS = [
    (0, 1000),
    (1000, 1400),
    (1400, 1800),
    (1800, 2200),
    (2200, 2600),
    (2600, 9999),
]

# Blunder threshold in centipawns (200 = same as Lichess)
BLUNDER_CUTOFF_CP = 200

# Sync parsed results to Drive every N games
SYNC_EVERY = 25_000

# Output file prefix
OUTPUT_PREFIX = "lichess_sampled"

## Setup
Install dependencies and mount Google Drive.

In [ ]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("python-chess")
install("zstandard")
install("pandas")
install("tqdm")

print("Dependencies installed.")

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Paths
DRIVE_ROOT = "/content/drive/MyDrive/Learning-document/Data Visualization/Project 2"
DRIVE_DATA_DIR = os.path.join(DRIVE_ROOT, "data")
VM_DATA_DIR = "/content/data"

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(VM_DATA_DIR, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Drive data:    {DRIVE_DATA_DIR}")
print(f"VM data dir:   {VM_DATA_DIR}")

## Download
Download the zstd-compressed PGN file from Lichess.

In [ ]:
def download_month(year, month):
    """Download zstd-compressed PGN file for a given month. Returns local path."""
    month_str = f"{year}-{month:02d}"
    filename = f"lichess_db_standard_rated_{month_str}.pgn.zst"
    url = f"https://database.lichess.org/standard/{filename}"
    local_path = f"/content/{filename}"

    if os.path.exists(local_path):
        print(f"File already exists: {local_path}")
        return local_path

    print(f"Downloading {url}...")
    print("This may take 10-30 minutes depending on file size.")
    urllib.request.urlretrieve(url, local_path)
    size_gb = os.path.getsize(local_path) / (1024**3)
    print(f"Downloaded: {size_gb:.1f} GB")
    return local_path

print("download_month() ready.")

## Streaming PGN Parser
Read the compressed PGN file line-by-line (no need to decompress to disk).
Extract both game metadata and move-level eval/clock data.

In [ ]:
import zstandard as zstd
import io
import chess
import chess.pgn
import re
from collections import defaultdict
from tqdm.notebook import tqdm
import pandas as pd
import time
import json
import urllib.request
import statistics

PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
}

def get_material(board):
    """Get (white_material, black_material) from board state."""
    wm = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.WHITE)
    bm = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.BLACK)
    return wm, bm

print("Libraries loaded.")

In [ ]:
def open_zst_stream(filepath):
    """Open a .zst file for streaming line-by-line reading."""
    dctx = zstd.ZstdDecompressor()
    with open(filepath, 'rb') as fh:
        stream = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(stream, encoding='utf-8')
        yield from text_stream


def read_games_from_stream(lines):
    """Yield individual game strings from a stream of PGN lines."""
    current_game = []
    for line in lines:
        line = line.rstrip('\n')
        if line == '' and current_game:
            has_movetext = any(not l.startswith('[') for l in current_game)
            if has_movetext:
                yield '\n'.join(current_game)
                current_game = []
            else:
                current_game.append(line)
        else:
            if line:
                current_game.append(line)
    if current_game:
        yield '\n'.join(current_game)


def parse_header(headers, key):
    """Safely extract a PGN header value."""
    try:
        return headers.get(key, None)
    except:
        return None


def parse_time_control(tc_str):
    """Parse time control like '300+0' or '600+5' into (base_seconds, increment)."""
    if not tc_str or '+' not in tc_str:
        return None, None
    try:
        parts = tc_str.split('+')
        return int(parts[0]), int(parts[1])
    except:
        return None, None


def extract_elos_from_text(game_text):
    """Quickly extract WhiteElo and BlackElo from raw PGN text."""
    white_elo = None
    black_elo = None
    for line in game_text.split('\n'):
        if line.startswith('[WhiteElo "'):
            try: white_elo = int(line.split('"')[1])
            except: pass
        elif line.startswith('[BlackElo "'):
            try: black_elo = int(line.split('"')[1])
            except: pass
        if white_elo is not None and black_elo is not None:
            break
    return white_elo, black_elo


print("Parser functions defined.")

## Collect & Process Games
Phase 1: Fast-scan PGN files for games with eval annotations, bucket by ELO, and sample evenly.
Phase 2: Full parse of sampled games to extract metadata, moves, and blunders.

In [ ]:
def save_all(games_metadata, move_records, blunder_records,
             piece_square_counts, material_by_ply,
             vm_dir, drive_dir, prefix):
    """Save all 5 CSVs to VM and sync to Drive."""
    for suffix, df_args in [
        ("games", pd.DataFrame(games_metadata)),
        ("moves", pd.DataFrame(move_records)),
        ("blunders", pd.DataFrame(blunder_records)),
    ]:
        path = os.path.join(vm_dir, f"{prefix}_{suffix}.csv")
        df_args.to_csv(path, index=False)

    # Piece squares (aggregated)
    psq_rows = [{'piece': k[0], 'square': k[1], 'is_white': k[2], 'period': k[3], 'count': v}
                for k, v in piece_square_counts.items()]
    pd.DataFrame(psq_rows).to_csv(os.path.join(vm_dir, f"{prefix}_piece_squares.csv"), index=False)

    # Material curve (aggregated averages)
    mat_rows = [{'period': k[0], 'ply': k[1],
                 'avg_white_material': round(v[0]/v[2], 2) if v[2] else 0,
                 'avg_black_material': round(v[1]/v[2], 2) if v[2] else 0,
                 'game_count': v[2]}
                for k, v in material_by_ply.items()]
    pd.DataFrame(mat_rows).to_csv(os.path.join(vm_dir, f"{prefix}_material_curve.csv"), index=False)

    # Sync to Drive
    for suffix in ["games", "moves", "blunders", "piece_squares", "material_curve"]:
        src = os.path.join(vm_dir, f"{prefix}_{suffix}.csv")
        dst = os.path.join(drive_dir, f"{prefix}_{suffix}.csv")
        if os.path.exists(src):
            shutil.copy2(src, dst)


def collect_sampled_games(filepath, buckets, period_label, elo_brackets, target_per_cell):
    """Phase 1: fast scan. Check '%eval' in raw text, bucket by ELO, collect game texts."""
    game_count = 0
    eval_count = 0
    start_time = time.time()

    for game_text in read_games_from_stream(open_zst_stream(filepath)):
        game_count += 1
        if '%eval' not in game_text:
            continue
        eval_count += 1

        white_elo, black_elo = extract_elos_from_text(game_text)
        if white_elo is None or black_elo is None:
            continue
        avg_elo = (white_elo + black_elo) / 2

        for bracket in elo_brackets:
            if bracket[0] <= avg_elo < bracket[1]:
                key = (period_label, bracket)
                if len(buckets[key]) < target_per_cell:
                    buckets[key].append((game_text, period_label))
                break

        if game_count % 500_000 == 0:
            total_collected = sum(len(v) for v in buckets.values())
            elapsed = time.time() - start_time
            print(f"  Scanned {game_count:,} ({game_count/elapsed:.0f}/s) | "
                  f"{eval_count:,} eval | collected {total_collected:,}")

        if all(len(buckets[(period_label, b)]) >= target_per_cell for b in elo_brackets):
            break

    total_collected = sum(len(v) for v in buckets.values())
    elapsed = time.time() - start_time
    print(f"  Scan: {game_count:,} in {elapsed:.1f}s, {eval_count:,} eval, collected {total_collected:,}")


def process_collected_games(game_data, sync_every=None,
                            vm_dir=None, drive_dir=None, prefix=None):
    """Phase 2: full parse of (game_text, period_label) tuples."""
    games_metadata = []
    move_records = []
    blunder_records = []
    piece_square_counts = defaultdict(int)
    material_by_ply = defaultdict(lambda: [0, 0, 0])

    game_count = 0
    start_time = time.time()
    total = len(game_data)

    for game_text, period_label in game_data:
        game = chess.pgn.read_game(io.StringIO(game_text))
        if game is None:
            continue
        game_count += 1
        headers = game.headers

        # --- Headers ---
        site = parse_header(headers, 'Site')
        white_elo = parse_header(headers, 'WhiteElo')
        black_elo = parse_header(headers, 'BlackElo')
        result = parse_header(headers, 'Result')
        time_control = parse_header(headers, 'TimeControl')
        termination = parse_header(headers, 'Termination')
        opening_eco = parse_header(headers, 'ECO')
        opening_name = parse_header(headers, 'Opening')
        date_str = parse_header(headers, 'UTCDate') or parse_header(headers, 'Date') or ''
        try:
            year = int(date_str.split('.')[0])
        except:
            year = None

        if result == '1-0': winner = 'white'
        elif result == '0-1': winner = 'black'
        elif result == '1/2-1/2': winner = 'draw'
        else: winner = None

        base_secs, increment = parse_time_control(time_control)

        victory_status = None
        if termination:
            if 'Normal' in termination:
                victory_status = 'mate' if 'checkmate' in termination.lower() else 'resign'
            elif 'Time' in termination: victory_status = 'timeout'
            elif 'Abandoned' in termination: victory_status = 'abandoned'
            elif 'Rules infraction' in termination: victory_status = 'rules_infraction'

        w_rat = int(white_elo) if white_elo and white_elo != '?' else None
        b_rat = int(black_elo) if black_elo and black_elo != '?' else None
        avg_elo = ((w_rat or 0) + (b_rat or 0)) / 2
        elo_bracket = None
        for br in ELO_BRACKETS:
            if br[0] <= avg_elo < br[1]:
                elo_bracket = f"{br[0]}-{br[1]}" if br[1] < 9999 else f"{br[0]}+"
                break

        game_type = None
        if base_secs is not None:
            total_est = base_secs + (increment or 0) * 80
            if total_est < 180: game_type = 'bullet'
            elif total_est < 480: game_type = 'blitz'
            elif total_est < 1500: game_type = 'rapid'
            else: game_type = 'classical'

        # --- Move-by-move ---
        moves_list = []
        board = game.board()
        node = game
        prev_eval = 0.0
        eval_changes = []
        sacrifice_count = 0
        white_castled = None
        black_castled = None
        has_eval = False
        has_clock = False
        white_mat, black_mat = get_material(board)
        ply = 0

        while node.variations:
            next_node = node.variation(0)
            move = next_node.move
            is_white = board.turn == chess.WHITE
            ply += 1

            # Piece info before push
            piece = board.piece_at(move.from_square)
            piece_name = chess.piece_name(piece.piece_type) if piece else 'unknown'
            dest = chess.square_name(move.to_square)
            san = board.san(move)
            is_capture = 'x' in san
            is_check = '+' in san or '#' in san
            moves_list.append(san)

            # Castling
            if san == 'O-O':
                if is_white: white_castled = ('K', ply)
                else: black_castled = ('K', ply)
            elif san == 'O-O-O':
                if is_white: white_castled = ('Q', ply)
                else: black_castled = ('Q', ply)

            # Eval + clock
            eval_cp = None
            eval_annotation = next_node.eval()
            if eval_annotation is not None:
                has_eval = True
                score = eval_annotation.white()
                if score.score() is not None:
                    eval_cp = score.score()
                elif score.mate() is not None:
                    eval_cp = 10000 if score.mate() > 0 else -10000

            clock_secs = next_node.clock()
            if clock_secs is not None:
                has_clock = True

            # Save material before push
            prev_wm, prev_bm = white_mat, black_mat

            # Push
            board.push(move)
            white_mat, black_mat = get_material(board)

            # Track material for ALL moves (aggregation)
            mat_key = (period_label, ply)
            material_by_ply[mat_key][0] += white_mat
            material_by_ply[mat_key][1] += black_mat
            material_by_ply[mat_key][2] += 1

            # Track piece-square for ALL moves (aggregation)
            piece_square_counts[(piece_name, dest, is_white, period_label)] += 1

            # Eval change
            eval_change = None
            if eval_cp is not None:
                eval_change = (eval_cp - prev_eval) if is_white else (prev_eval - eval_cp)
                eval_changes.append(eval_change)

            # Sacrifice: material loss >= 2 without eval crash
            is_sacrifice = False
            material_loss = (prev_wm - white_mat) if is_white else (prev_bm - black_mat)
            if material_loss >= 2 and eval_change is not None and eval_change > -50:
                is_sacrifice = True
                sacrifice_count += 1

            # Record eval moves only
            if eval_cp is not None:
                move_records.append({
                    'game_idx': game_count - 1,
                    'ply': ply,
                    'is_white': is_white,
                    'piece': piece_name,
                    'dest_square': dest,
                    'is_capture': is_capture,
                    'is_check': is_check,
                    'eval_cp': eval_cp,
                    'eval_change_for_player': eval_change,
                    'clock_secs': clock_secs,
                    'white_material': white_mat,
                    'black_material': black_mat,
                    'is_sacrifice': is_sacrifice,
                })

                # Blunder
                if eval_change is not None and eval_change <= -BLUNDER_CUTOFF_CP:
                    was_winning = (is_white and prev_eval > 200) or (not is_white and prev_eval < -200)
                    still_winning = (is_white and eval_cp > 200) or (not is_white and eval_cp < -200)
                    if not (was_winning and still_winning):
                        blunder_records.append({
                            'game_idx': game_count - 1,
                            'ply': ply,
                            'is_white': is_white,
                            'eval_change_cp': eval_change,
                            'clock_secs_remaining': clock_secs,
                            'player_elo': w_rat if is_white else b_rat,
                            'period': period_label,
                        })

                prev_eval = eval_cp

            node = next_node

        # --- Game metadata ---
        eval_vol = statistics.stdev(eval_changes) if len(eval_changes) > 1 else 0

        games_metadata.append({
            'game_id': site.split('/')[-1] if site else None,
            'year': year,
            'period': period_label,
            'white_rating': w_rat,
            'black_rating': b_rat,
            'avg_elo': avg_elo if avg_elo > 0 else None,
            'elo_bracket': elo_bracket,
            'winner': winner,
            'victory_status': victory_status,
            'game_type': game_type,
            'opening_eco': opening_eco,
            'opening_name': opening_name,
            'first_move': moves_list[0] if moves_list else None,
            'first_10_moves': ','.join(moves_list[:10]),
            'game_length': len(moves_list),
            'white_castled': white_castled[0] if white_castled else 'none',
            'white_castling_ply': white_castled[1] if white_castled else 0,
            'black_castled': black_castled[0] if black_castled else 'none',
            'black_castling_ply': black_castled[1] if black_castled else 0,
            'sacrifice_count': sacrifice_count,
            'eval_volatility': round(eval_vol, 2),
            'has_eval': has_eval,
            'has_clock': has_clock,
        })

        # Periodic sync
        if sync_every and game_count % sync_every == 0:
            elapsed = time.time() - start_time
            print(f"  Parsed {game_count:,}/{total:,} ({game_count/elapsed:.0f}/s) - syncing...")
            save_all(games_metadata, move_records, blunder_records,
                     piece_square_counts, material_by_ply,
                     vm_dir, drive_dir, prefix)

    # Final save
    if vm_dir:
        save_all(games_metadata, move_records, blunder_records,
                 piece_square_counts, material_by_ply,
                 vm_dir, drive_dir, prefix)

    elapsed = time.time() - start_time
    print(f"\nDone: {game_count:,}/{total:,} in {elapsed:.1f}s")

    return games_metadata, move_records, blunder_records, piece_square_counts, material_by_ply

In [ ]:
import itertools

elo_brackets = ELO_BRACKETS
num_cells = len(PERIODS) * len(elo_brackets)
target_per_cell = TOTAL_GAMES // num_cells

buckets = {}
for period in PERIODS:
    for bracket in elo_brackets:
        buckets[(period["label"], bracket)] = []

months = MONTH if MONTH is not None else list(range(1, 13))
if not isinstance(months, list):
    months = [months]

# Phase 1: Collect sampled games across periods and ELO brackets
for period in PERIODS:
    label = period["label"]
    period_full = all(len(buckets[(label, b)]) >= target_per_cell for b in elo_brackets)
    if period_full:
        continue

    for y in period["years"]:
        for m in months:
            period_full = all(len(buckets[(label, b)]) >= target_per_cell for b in elo_brackets)
            if period_full:
                break

            month_str = f"{y}-{m:02d}"
            print(f"\n{'='*60}")
            print(f"Scanning {month_str} [{label}]")
            print(f"{'='*60}")

            local_path = download_month(y, m)
            collect_sampled_games(local_path, buckets, label, elo_brackets, target_per_cell)

            if os.path.exists(local_path):
                os.remove(local_path)
                print(f"Deleted {local_path}")
        if period_full:
            break

# Collection summary
print(f"\n{'='*60}")
print(f"Collection summary (target per cell: {target_per_cell:,}):")
for period in PERIODS:
    label = period["label"]
    print(f"\n  [{label}]")
    for bracket in elo_brackets:
        lo, hi = bracket
        b_label = f"{lo}-{hi}" if hi < 9999 else f"{lo}+"
        count = len(buckets[(label, bracket)])
        print(f"    ELO {b_label:>9}: {count:,}")
total = sum(len(v) for v in buckets.values())
print(f"\n  Total: {total:,}")
print(f"{'='*60}")

# Phase 2: Parse collected games
all_game_data = []
for period in PERIODS:
    for bracket in elo_brackets:
        all_game_data.extend(buckets[(period["label"], bracket)])
del buckets

print(f"\nParsing {len(all_game_data):,} games...")
games_metadata, move_records, blunder_records, piece_squares, material_curve = process_collected_games(
    all_game_data,
    sync_every=SYNC_EVERY,
    vm_dir=VM_DATA_DIR,
    drive_dir=DRIVE_DATA_DIR,
    prefix=OUTPUT_PREFIX,
)

print(f"\nResults:")
print(f"  Games: {len(games_metadata):,}")
print(f"  Moves (eval): {len(move_records):,}")
print(f"  Blunders: {len(blunder_records):,}")
print(f"  Piece-square entries: {len(piece_squares):,}")
print(f"  Material curve points: {len(material_curve):,}")

del all_game_data

## Save to Google Drive

In [ ]:
# Create DataFrames for preview
df_games = pd.DataFrame(games_metadata)
df_moves = pd.DataFrame(move_records)
df_blunders = pd.DataFrame(blunder_records)
df_psq = pd.DataFrame([{'piece': k[0], 'square': k[1], 'is_white': k[2], 'period': k[3], 'count': v}
                        for k, v in piece_squares.items()])
df_mat = pd.DataFrame([{'period': k[0], 'ply': k[1],
                         'avg_white_material': round(v[0]/v[2], 2) if v[2] else 0,
                         'avg_black_material': round(v[1]/v[2], 2) if v[2] else 0,
                         'game_count': v[2]}
                        for k, v in material_curve.items()])

print(f"Games: {df_games.shape}")
print(f"Moves: {df_moves.shape}")
print(f"Blunders: {df_blunders.shape}")
print(f"Piece squares: {df_psq.shape}")
print(f"Material curve: {df_mat.shape}")

print(f"\n=== Games by period ===")
print(df_games['period'].value_counts().to_string())

print(f"\n=== First move distribution ===")
print(df_games['first_move'].value_counts().head(5).to_string())

print(f"\n=== Game length stats ===")
print(df_games['game_length'].describe().to_string())

In [ ]:
# Verify all saved files on Drive
print("Files on Drive:")
for f in sorted(os.listdir(DRIVE_DATA_DIR)):
    if f.startswith(OUTPUT_PREFIX) and f.endswith(".csv"):
        path = os.path.join(DRIVE_DATA_DIR, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  {f} ({size_mb:.1f} MB)")

## Summary Statistics

In [ ]:
print("=== Summary ===")
print(f"Total games: {len(df_games):,}")

print(f"\n--- Period distribution ---")
print(df_games['period'].value_counts().to_string())

print(f"\n--- ELO bracket distribution ---")
print(df_games['elo_bracket'].value_counts().to_string())

print(f"\n--- First move by period ---")
print(pd.crosstab(df_games['first_move'], df_games['period']).to_string())

print(f"\n--- Game length by period ---")
for p in df_games['period'].unique():
    subset = df_games[df_games['period'] == p]['game_length']
    print(f"  {p}: mean={subset.mean():.1f}, median={subset.median():.0f}")

print(f"\n--- Sacrifice rate by period ---")
for p in df_games['period'].unique():
    subset = df_games[df_games['period'] == p]
    rate = subset['sacrifice_count'].mean()
    print(f"  {p}: avg {rate:.2f} sacrifices/game")

print(f"\n--- Blunder rate by period ---")
if not df_blunders.empty:
    for p in df_blunders['period'].unique():
        n_blunders = len(df_blunders[df_blunders['period'] == p])
        n_games = len(df_games[df_games['period'] == p])
        print(f"  {p}: {n_blunders:,} blunders in {n_games:,} games ({n_blunders/n_games:.1f}/game)")

print(f"\n--- Eval volatility by period ---")
for p in df_games['period'].unique():
    vol = df_games[df_games['period'] == p]['eval_volatility'].mean()
    print(f"  {p}: avg volatility = {vol:.1f}cp")